# Query Classification Pipeline

We're building a query intent classifier for a search engine. The classifier will be used in production for tasks like result reranking.

We have search queries with their top search results (title, snippet, URL).  
Goal: classify each query into intent categories — `sports`, `technology`, or `shopping`.

## 1. Setup & Configuration

In [ ]:
import json
from collections import Counter

import torch
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

LABELS = ["sports", "technology", "shopping"]

INPUT_FILE = "data/queries_with_results.json"

## 2. Load Data

In [ ]:
with open(INPUT_FILE) as f:
    data = json.load(f)

print(f"Loaded {len(data)} queries")
print(f"\nSample entry:")
print(json.dumps(data[0], indent=2))

## 3. Label Queries

Assign a category to each query using keyword matching.

In [ ]:
KEYWORD_MAP = {
    "sports": [
        "football", "soccer", "basketball", "nba", "nfl", "cricket", "tennis",
        "golf", "baseball", "mlb", "boxing", "surfing", "olympics", "champion",
        "league", "match", "game", "score", "race", "f1", "draft", "tournament",
        "premier", "world cup", "esports", "fitness", "gym", "running", "yoga",
    ],
    "technology": [
        "python", "javascript", "react", "vue", "kubernetes", "docker", "cloud",
        "machine learning", "ai", "gpu", "rtx", "raspberry", "cybersecurity",
        "vpn", "data engineering", "iphone", "laptop", "monitor", "framework",
        "certification", "software",
    ],
    "shopping": [
        "buy", "deal", "sale", "cheap", "discount", "price", "shop", "coupon",
        "black friday", "prime day", "clearance", "bundle",
    ],
}


def classify_query(query):
    query_lower = query.lower()
    scores = {}
    for label, keywords in KEYWORD_MAP.items():
        scores[label] = sum(1 for kw in keywords if kw in query_lower)

    best_label = max(scores, key=scores.get)
    if scores[best_label] == 0:
        return None
    return best_label

In [ ]:
queries = []
labels = []
unlabeled = 0

for item in data:
    label = classify_query(item["query"])
    if label is None:
        unlabeled += 1
        continue
    queries.append(item["query"])
    labels.append(LABELS.index(label))

label_dist = Counter(LABELS[l] for l in labels)
print(f"Labeled: {len(queries)}, Unlabeled: {unlabeled}")
print(f"Label distribution: {dict(label_dist)}")

## 4. Prepare Dataset & Tokenize

In [ ]:
class QueryDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encodings = tokenizer(queries, truncation=True, padding=True, max_length=64)
dataset = QueryDataset(encodings, labels)

print(f"Dataset size: {len(dataset)}")
print(f"Sample tokens: {tokenizer.decode(encodings['input_ids'][0])}")

## 5. Train Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(LABELS)
)

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

## 6. Evaluate

In [ ]:
predictions = trainer.predict(dataset)
predicted_labels = predictions.predictions.argmax(axis=1)

acc = accuracy_score(labels, predicted_labels)
print(f"Accuracy: {acc:.4f}")